# Phase 6: Paper Trading Preparation

Paper trading is the bridge between backtesting and live trading. It validates your system in realistic conditions without risking real money.

This notebook covers:
1. **Broker Simulator**: Order execution with slippage
2. **Logging System**: Audit trail for all activity
3. **Performance Monitoring**: Real-time tracking
4. **Transition Checklist**: What to verify before live trading

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

from datetime import datetime, timedelta

from src.broker.simulator import BrokerSimulator, BrokerConfig
from src.broker.orders import OrderSide, OrderType
from src.monitoring.performance import PerformanceMonitor, TradeRecord
from src.utils.logger import TradingLogger

print("Setup complete!")

Setup complete!


## 1. Broker Simulator

The broker simulator mimics a real forex broker:
- **Order types**: Market, Limit, Stop
- **Slippage**: 0.5-2 pips (price impact)
- **Commission**: $7 per lot (100k units)
- **Margin**: 50:1 leverage
- **Swap**: Overnight interest credit/debit

In [2]:
config = BrokerConfig(
    initial_balance=100000,
    leverage=50.0,
    min_slippage_pips=0.5,
    max_slippage_pips=1.0,
    commission_per_lot=7.0,
    order_latency_ms=(50, 150),
)

broker = BrokerSimulator(config)

print(f"Initial balance: ${broker.balance:,.2f}")
print(f"Leverage: {config.leverage}:1")

Initial balance: $100,000.00
Leverage: 50.0:1


In [3]:
# Place a market order
order = broker.submit_market_order("USD/JPY", OrderSide.BUY, lots=0.1)
print(f"Order submitted: {order.order_id}")
print(f"Status: {order.status.name}")

Order submitted: 2671e270
Status: SUBMITTED


In [4]:
# Execute at current price
fills = broker.execute_orders({"USD/JPY": 155.50})

if fills:
    fill = fills[0]
    print(f"Order filled!")
    print(f"  Price: {fill.price:.2f}")
    print(f"  Slippage: {fill.slippage:.2f} pips")
    print(f"  Commission: ${fill.commission:.2f}")

Order filled!
  Price: 155.51
  Slippage: 0.55 pips
  Commission: $0.70


In [5]:
# Check position
pos = broker.get_position("USD/JPY")
if pos:
    print(f"Position: {pos.side.name} {pos.quantity:,.0f} units @ {pos.entry_price:.2f}")
    
# Update with new price
broker.execute_orders({"USD/JPY": 156.00})
pos = broker.get_position("USD/JPY")
if pos:
    print(f"Unrealized P&L: ${pos.unrealized_pnl:,.2f}")

Position: BUY 10,000 units @ 155.51
Unrealized P&L: $4,945.25


In [6]:
# Check account state
state = broker.get_account_state()
print(f"\nAccount State:")
print(f"  Balance: ${state.balance:,.2f}")
print(f"  Equity: ${state.equity:,.2f}")
print(f"  Margin used: ${state.margin_used:,.2f}")
print(f"  Free margin: ${state.free_margin:,.2f}")
print(f"  Margin level: {state.margin_level:.1f}%")


Account State:
  Balance: $99,999.30
  Equity: $104,944.55
  Margin used: $31,200.00
  Free margin: $73,744.55
  Margin level: 336.4%


In [7]:
# Close position
close_order = broker.close_position("USD/JPY")
broker.execute_orders({"USD/JPY": 156.20})

state = broker.get_account_state()
print(f"Position closed!")
print(f"Final balance: ${state.balance:,.2f}")
print(f"Profit: ${state.balance - config.initial_balance:,.2f}")

Position closed!
Final balance: $104,943.85
Profit: $4,943.85


## 2. Performance Monitoring

Real-time tracking of:
- Equity curve
- Drawdown (current and maximum)
- Win rate and profit factor
- Alerts on threshold breaches

In [8]:
def alert_handler(alert_type, message, value):
    print(f"[ALERT] {alert_type}: {message}")

monitor = PerformanceMonitor(
    initial_equity=100000,
    alert_callback=alert_handler,
    max_drawdown_alert=0.05,  # Alert at 5% drawdown
    daily_loss_alert=0.02,    # Alert at 2% daily loss
)

print("Performance monitor initialized")

Performance monitor initialized


In [9]:
# Simulate equity updates
equities = [100000, 101000, 102500, 101500, 103000, 98000, 99500, 104000]

for i, eq in enumerate(equities):
    daily_pnl = eq - equities[i-1] if i > 0 else 0
    monitor.update(
        equity=eq,
        balance=eq,
        unrealized_pnl=0,
        daily_pnl=daily_pnl,
        positions_count=0,
    )
    snapshot = monitor.get_snapshot()
    print(f"Equity: ${eq:,} | Drawdown: {snapshot.drawdown:.1%} | Max DD: {snapshot.max_drawdown:.1%}")

Equity: $100,000 | Drawdown: 0.0% | Max DD: 0.0%
Equity: $101,000 | Drawdown: 0.0% | Max DD: 0.0%
Equity: $102,500 | Drawdown: 0.0% | Max DD: 0.0%
Equity: $101,500 | Drawdown: 1.0% | Max DD: 1.0%
Equity: $103,000 | Drawdown: 0.0% | Max DD: 1.0%
[ALERT] DAILY_LOSS: Daily loss reached 5.0%
Equity: $98,000 | Drawdown: 4.9% | Max DD: 4.9%
Equity: $99,500 | Drawdown: 3.4% | Max DD: 4.9%
Equity: $104,000 | Drawdown: 0.0% | Max DD: 4.9%


In [10]:
# Record some trades
trades = [
    (150.0, 151.5, 500, True),   # Win
    (152.0, 151.0, -300, False), # Loss
    (150.5, 152.0, 450, True),   # Win
    (151.0, 150.0, -350, False), # Loss
    (150.0, 153.0, 900, True),   # Win
]

for entry, exit_p, pnl, is_win in trades:
    trade = TradeRecord(
        timestamp=datetime.now(),
        pair="USD/JPY",
        side="BUY",
        entry_price=entry,
        exit_price=exit_p,
        quantity=10000,
        pnl=pnl,
        pnl_pct=pnl/100000,
        duration=timedelta(hours=5),
        swap_earned=1.37 if is_win else 0,
    )
    monitor.record_trade(trade)

stats = monitor.get_trade_stats()
print(f"\nTrade Statistics:")
print(f"  Total trades: {stats['total_trades']}")
print(f"  Win rate: {stats['win_rate']:.1%}")
print(f"  Profit factor: {stats['profit_factor']:.2f}")
print(f"  Avg win: ${stats['avg_win']:.2f}")
print(f"  Avg loss: ${stats['avg_loss']:.2f}")


Trade Statistics:
  Total trades: 5
  Win rate: 60.0%
  Profit factor: 2.85
  Avg win: $616.67
  Avg loss: $-325.00


## 3. Order Types

The broker simulator supports different order types:
- **Market**: Execute immediately at current price
- **Limit**: Execute only at target price or better
- **Stop**: Trigger when price reaches stop level

In [11]:
# Fresh broker for order type demo
broker2 = BrokerSimulator(BrokerConfig(order_latency_ms=(0, 0)))

# Limit order - buy if price drops to 154
limit_order = broker2.submit_limit_order("USD/JPY", OrderSide.BUY, 0.1, limit_price=154.0)
print(f"Limit order: Buy if price drops to 154.00")

# Current price is 155 - limit not hit
broker2.execute_orders({"USD/JPY": 155.0})
print(f"  Price at 155.0 -> Status: {limit_order.status.name}")

# Price drops to 153.5 - limit hit!
broker2.execute_orders({"USD/JPY": 153.5})
print(f"  Price at 153.5 -> Status: {limit_order.status.name}")

Limit order: Buy if price drops to 154.00
  Price at 155.0 -> Status: SUBMITTED
  Price at 153.5 -> Status: FILLED


In [12]:
# Stop order - sell if price drops to 152 (stop loss)
broker3 = BrokerSimulator(BrokerConfig(order_latency_ms=(0, 0)))

# Open position first
broker3.submit_market_order("USD/JPY", OrderSide.BUY, 0.1)
broker3.execute_orders({"USD/JPY": 155.0})

# Set stop loss
stop_order = broker3.submit_stop_order("USD/JPY", OrderSide.SELL, 0.1, stop_price=152.0)
print(f"Stop order: Sell if price drops to 152.00")

# Price at 154 - stop not hit
broker3.execute_orders({"USD/JPY": 154.0})
print(f"  Price at 154.0 -> Stop status: {stop_order.status.name}")

# Price drops to 151.5 - stop triggered!
broker3.execute_orders({"USD/JPY": 151.5})
print(f"  Price at 151.5 -> Stop status: {stop_order.status.name}")

Stop order: Sell if price drops to 152.00
  Price at 154.0 -> Stop status: SUBMITTED
  Price at 151.5 -> Stop status: FILLED


## 4. Paper Trading Checklist

Before transitioning to live trading, verify:

### System Validation
- [ ] Paper trading ran 4+ weeks without critical errors
- [ ] Win rate within 20% of backtested expectations
- [ ] Maximum drawdown stayed within limits
- [ ] All circuit breakers functioned correctly
- [ ] Logging captured all necessary audit information

### Broker Integration
- [ ] Broker selected with API access
- [ ] API credentials secured (encrypted)
- [ ] Connection tested (read-only first)
- [ ] Rate limiting implemented
- [ ] Reconnection logic tested

### Risk Management
- [ ] Position sizing validated
- [ ] Stop losses functional
- [ ] Circuit breakers tested
- [ ] Emergency kill switch ready
- [ ] Only trading with money you can afford to lose

In [13]:
print("="*60)
print("PROJECT COMPLETE!")
print("="*60)
print("""
All 6 phases have been implemented:

1. Foundations & Synthetic Data
   - GBM price generator
   - Vasicek interest rate model
   - OHLCV + swap data generation

2. Basic Strategy & Backtesting
   - Carry trade strategy (swap + SMA + RSI)
   - Backtest engine with metrics
   - Portfolio management

3. Optimization & Visualization
   - Grid search optimizer
   - Walk-forward analysis
   - Interactive charts

4. Real Data & Refinement
   - Yahoo Finance integration
   - Data quality validation
   - Strategy recalibration

5. Advanced Risk Management
   - Position sizing (Kelly, FF, Vol)
   - ATR-based stops
   - VaR/CVaR analysis
   - Circuit breakers

6. Paper Trading Preparation
   - Broker simulator
   - Logging system
   - Performance monitoring
   - Transition documentation

This is an EDUCATIONAL project - no real money was used.
""")

PROJECT COMPLETE!

All 6 phases have been implemented:

1. Foundations & Synthetic Data
   - GBM price generator
   - Vasicek interest rate model
   - OHLCV + swap data generation

2. Basic Strategy & Backtesting
   - Carry trade strategy (swap + SMA + RSI)
   - Backtest engine with metrics
   - Portfolio management

3. Optimization & Visualization
   - Grid search optimizer
   - Walk-forward analysis
   - Interactive charts

4. Real Data & Refinement
   - Yahoo Finance integration
   - Data quality validation
   - Strategy recalibration

5. Advanced Risk Management
   - Position sizing (Kelly, FF, Vol)
   - ATR-based stops
   - VaR/CVaR analysis
   - Circuit breakers

6. Paper Trading Preparation
   - Broker simulator
   - Logging system
   - Performance monitoring
   - Transition documentation

This is an EDUCATIONAL project - no real money was used.

